In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt

# 1. Create a set of cities with random coordinates
num_cities = 15
random.seed(42)
np.random.seed(42)

cities = {i: (random.uniform(0, 100), random.uniform(0, 100)) for i in range(num_cities)}

# Calculate Euclidean distance between two cities
def distance(city1, city2):
    return np.sqrt((city1[0] - city2[0])**2 + (city1[1] - city2[1])**2)

# Calculate total distance of a route
def route_distance(route):
    dist = 0
    for i in range(len(route)):
        dist += distance(cities[route[i]], cities[route[(i + 1) % len(route)]])
    return dist

# 2. Genetic Algorithm Parameters
pop_size = 100
elite_size = 10
mutation_rate = 0.02
generations = 500

# Create initial population
def create_population(size):
    population = []
    for _ in range(size):
        route = list(cities.keys())
        random.shuffle(route)
        population.append(route)
    return population

# Rank routes based on fitness (1 / route_distance)
def rank_routes(population):
    fitness_results = {}
    for i in range(len(population)):
        fitness_results[i] = 1 / float(route_distance(population[i]))
    return sorted(fitness_results.items(), key=lambda x: x[1], reverse=True)

# Selection using Tournament Selection
def selection(population, ranked_pop, elite_size):
    selection_results = []
    # Keep elite individuals
    for i in range(elite_size):
        selection_results.append(population[ranked_pop[i][0]])

    # Tournament selection for the rest
    for _ in range(len(population) - elite_size):
        candidates = random.sample(ranked_pop, 5)
        best_candidate = max(candidates, key=lambda x: x[1])
        selection_results.append(population[best_candidate[0]])

    return selection_results

# Crossover: Ordered Crossover (OX) for TSP
def crossover(parent1, parent2):
    child = [None] * len(parent1)

    start, end = sorted(random.sample(range(len(parent1)), 2))

    # Copy slice from parent1 to child
    for i in range(start, end + 1):
        child[i] = parent1[i]

    # Fill remaining spots with elements from parent2 in order
    p2_index = 0
    for i in range(len(child)):
        if child[i] is None:
            while parent2[p2_index] in child:
                p2_index += 1
            child[i] = parent2[p2_index]

    return child

# Create offspring generation
def breed_population(matingpool, elite_size):
    children = []
    # Keep elite
    for i in range(elite_size):
        children.append(matingpool[i])

    # Crossover to create remaining offspring
    for i in range(elite_size, len(matingpool)):
        parent1 = matingpool[i]
        parent2 = matingpool[len(matingpool) - i - 1]
        children.append(crossover(parent1, parent2))

    return children

# Mutation: Swap Mutation
def mutate(individual, mutation_rate):
    for swapped in range(len(individual)):
        if random.random() < mutation_rate:
            swap_with = int(random.random() * len(individual))

            city1 = individual[swapped]
            city2 = individual[swap_with]

            individual[swapped] = city2
            individual[swap_with] = city1
    return individual

def mutate_population(population, mutation_rate, elite_size):
    mutated_pop = []
    # Do not mutate the elite individuals
    for i in range(elite_size):
        mutated_pop.append(population[i])

    for i in range(elite_size, len(population)):
        mutated_pop.append(mutate(population[i], mutation_rate))
    return mutated_pop

# Produce the next generation
def next_generation(current_gen, elite_size, mutation_rate):
    ranked_pop = rank_routes(current_gen)
    selected_pop = selection(current_gen, ranked_pop, elite_size)
    children = breed_population(selected_pop, elite_size)
    next_gen = mutate_population(children, mutation_rate, elite_size)
    return next_gen

# Run Genetic Algorithm
def genetic_algorithm(generations, pop_size, elite_size, mutation_rate):
    pop = create_population(pop_size)
    progress = []
    progress.append(1 / rank_routes(pop)[0][1])

    for i in range(generations):
        pop = next_generation(pop, elite_size, mutation_rate)
        progress.append(1 / rank_routes(pop)[0][1])

    best_route_index = rank_routes(pop)[0][0]
    best_route = pop[best_route_index]
    best_distance = 1 / rank_routes(pop)[0][1]

    return best_route, best_distance, progress

# 3. Execution & Visualization
best_route, best_distance, progress = genetic_algorithm(generations, pop_size, elite_size, mutation_rate)
print(f"Initial Distance: {progress[0]:.2f}")
print(f"Final Best Distance: {best_distance:.2f}")
print(f"Best Route: {best_route}")

# Plot optimization progress
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(progress)
plt.ylabel('Distance')
plt.xlabel('Generation')
plt.title('Optimization Progress over Generations')

# Plot best route
plt.subplot(1, 2, 2)
x = [cities[city][0] for city in best_route] + [cities[best_route[0]][0]]
y = [cities[city][1] for city in best_route] + [cities[best_route[0]][1]]
plt.plot(x, y, 'co-')
plt.scatter([cities[city][0] for city in cities], [cities[city][1] for city in cities], color='red')
for city, coords in cities.items():
    plt.annotate(str(city), (coords[0]+1, coords[1]+1))
plt.title('Best Reconstructed Route')
plt.tight_layout()
plt.show()